# Lecture 9 Demo: Hybrid Quantum-Classical Neural Networks

EVA: Quantum Machine Learning | ZHAW | Pavel Sulimov

---

This notebook accompanies Lecture 9. It demonstrates:

1. Building a minimal hybrid QNN with PennyLane and PyTorch (`TorchLayer`)
2. Comparing architecture patterns: Q-C, C-Q-C, and re-uploading
3. Encoding strategies in practice: angle vs re-uploading on 2D synthetic data
4. Training a hybrid model and inspecting gradient flow
5. Visualizing decision boundaries and comparing against a classical baseline

Convention: all qubits start in $\lvert 0\rangle$ by default.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import torch
import torch.nn as nn
import torch.optim as optim

import pennylane as qml
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

---
## Demo 1. Minimal hybrid QNN with TorchLayer

We build the simplest viable hybrid model for binary classification on 2D data:

$$
x \in \mathbb{R}^2
\xrightarrow{\text{angle enc.}} \text{PQC}(\theta)
\xrightarrow{\langle Z_0\rangle}
\sigma(\cdot) \to \hat{y}
$$

This is the Q-C pattern: a quantum circuit followed by a sigmoid output.

### Circuit design

Encoding: angle embedding with $R_y(\pi \tilde{x}_i)$ on each qubit, after scaling features to $[0,1]$.
Ansatz: strongly entangling layers with $L=2$ repetitions.
Output: $\langle Z_0\rangle \in [-1, 1]$, passed through a sigmoid.

In [ ]:
n_qubits = 2
n_layers = 2

dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit(inputs, weights):
    """Binary classifier PQC.

    Args:
        inputs: shape (n_qubits,) -- pre-scaled features in [0, 1]
        weights: shape (n_layers, n_qubits, 3) -- trainable rotation angles

    Returns:
        Expectation value of PauliZ on qubit 0.
    """
    qml.AngleEmbedding(inputs * np.pi, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


# Inspect the circuit structure for a single input
dummy_inputs  = torch.zeros(n_qubits)
dummy_weights = torch.zeros(n_layers, n_qubits, 3)
print(qml.draw(circuit)(dummy_inputs, dummy_weights))

In [ ]:
# Wrap in a TorchLayer and compose a minimal hybrid model
weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(circuit, weight_shapes)

# Q-C model: quantum circuit -> sigmoid
model_qc = nn.Sequential(
    qlayer,              # input: (batch, 2)  output: (batch, 1)
    nn.Sigmoid(),
)

# Verify a forward pass
x_test = torch.rand(5, n_qubits)
print("Input shape: ", x_test.shape)
print("Output shape:", model_qc(x_test).shape)
print("Output values:", model_qc(x_test).detach().numpy().ravel())

### Data: moons dataset

We use `make_moons` with added noise. Features are scaled to $[0, 1]$ before angle encoding
to avoid aliasing from the $4\pi$-periodicity of $R_y$.

In [ ]:
X_raw, y_raw = make_moons(n_samples=200, noise=0.2, random_state=42)

scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_raw, test_size=0.25, random_state=42
)

X_train = torch.tensor(X_tr, dtype=torch.float32)
y_train = torch.tensor(y_tr, dtype=torch.float32)
X_test  = torch.tensor(X_te, dtype=torch.float32)
y_test  = torch.tensor(y_te, dtype=torch.float32)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*X_scaled[y_raw == 0].T, s=18, label="class 0", alpha=0.7)
ax.scatter(*X_scaled[y_raw == 1].T, s=18, label="class 1", alpha=0.7)
ax.set_title("Moons dataset (scaled to [0,1])")
ax.set_xlabel("$\\tilde{x}_1$")
ax.set_ylabel("$\\tilde{x}_2$")
ax.legend()
plt.tight_layout()
plt.show()

### Training loop

Standard PyTorch training. No special handling is needed for the quantum layer;
`TorchLayer` registers quantum parameters as `nn.Parameter` and `loss.backward()`
computes gradients via the backprop-mode differentiation.

In [ ]:
def train_model(model, X_train, y_train, n_epochs=60, lr=0.01):
    """Generic training loop for binary classification."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    history = []

    for epoch in range(n_epochs):
        optimizer.zero_grad()
        preds = model(X_train).squeeze()
        loss = loss_fn(preds, y_train)
        loss.backward()
        optimizer.step()
        history.append(loss.item())

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch + 1:3d}  loss = {loss.item():.4f}")

    return history


print("Training Q-C model (quantum circuit -> sigmoid):")
history_qc = train_model(model_qc, X_train, y_train, n_epochs=80, lr=0.02)

In [ ]:
# Evaluate on test set
with torch.no_grad():
    probs_qc = model_qc(X_test).squeeze().numpy()
    preds_qc = (probs_qc >= 0.5).astype(int)

acc_qc = accuracy_score(y_te, preds_qc)
print(f"Q-C model test accuracy: {acc_qc:.3f}")

plt.figure(figsize=(5, 3))
plt.plot(history_qc)
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Q-C training curve")
plt.tight_layout()
plt.show()

---
## Demo 2. Architecture comparison: Q-C vs C-Q-C vs logistic regression

We now build three models and compare them on the same moons split:

1. **Q-C**: quantum circuit $\to$ sigmoid (from Demo 1)
2. **C-Q-C** (sandwich): `Linear(2,2) + Tanh` $\to$ quantum circuit $\to$ `Linear(2,1) + Sigmoid`
3. **Logistic regression** (classical baseline, always run this)

The C-Q-C model adds a classical encoder before the quantum layer to learn a useful
feature rotation, and a classical linear head after to combine the observable values.

In [ ]:
# Multi-observable circuit: returns expectation value on all qubits
@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit_multi(inputs, weights):
    """Returns <Z> on each qubit.

    Args:
        inputs: shape (n_qubits,)
        weights: shape (n_layers, n_qubits, 3)

    Returns:
        List of n_qubits expectation values.
    """
    qml.AngleEmbedding(inputs * np.pi, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


qlayer_multi = qml.qnn.TorchLayer(
    circuit_multi, {"weights": (n_layers, n_qubits, 3)}
)


class CQC(nn.Module):
    """C-Q-C hybrid: classical encoder -> quantum -> classical head."""

    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(2, n_qubits), nn.Tanh())
        self.qnode   = qlayer_multi
        self.head    = nn.Sequential(nn.Linear(n_qubits, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.encoder(x)  # (batch, n_qubits)
        # re-scale into [0,1] after tanh
        x = (x + 1.0) / 2.0
        x = self.qnode(x)    # (batch, n_qubits)
        return self.head(x)  # (batch, 1)


model_cqc = CQC()

# Sanity check forward pass
with torch.no_grad():
    print("C-Q-C output shape:", model_cqc(X_train[:4]).shape)

In [ ]:
print("Training C-Q-C model:")
history_cqc = train_model(model_cqc, X_train, y_train, n_epochs=80, lr=0.02)

with torch.no_grad():
    preds_cqc = (model_cqc(X_test).squeeze().numpy() >= 0.5).astype(int)
acc_cqc = accuracy_score(y_te, preds_cqc)
print(f"C-Q-C model test accuracy: {acc_cqc:.3f}")

In [ ]:
# Classical baseline
lr_clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
acc_lr = lr_clf.score(X_te, y_te)
print(f"Logistic regression test accuracy: {acc_lr:.3f}")

print("\nSummary:")
print(f"  Logistic regression : {acc_lr:.3f}")
print(f"  Q-C quantum model   : {acc_qc:.3f}")
print(f"  C-Q-C hybrid        : {acc_cqc:.3f}")

### Decision boundaries

Visualizing decision boundaries reveals what each model has learned.
The classical head in C-Q-C can smooth the boundary even when the quantum
circuit output is noisy.

In [ ]:
def plot_decision_boundary(model_fn, X, y, ax, title, is_torch=True):
    """Plot decision boundary for a binary classifier."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.05, X[:, 0].max() + 0.05
    y_min, y_max = X[:, 1].min() - 0.05, X[:, 1].max() + 0.05
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    if is_torch:
        with torch.no_grad():
            Z = model_fn(torch.tensor(grid, dtype=torch.float32)).squeeze().numpy()
    else:
        Z = model_fn(grid)

    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=50, cmap="RdBu", alpha=0.7)
    ax.scatter(*X[y == 0].T, s=14, c="royalblue", edgecolors="k", linewidths=0.3)
    ax.scatter(*X[y == 1].T, s=14, c="tomato",    edgecolors="k", linewidths=0.3)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


fig, axes = plt.subplots(1, 3, figsize=(13, 4))

plot_decision_boundary(
    lambda x: torch.tensor(lr_clf.predict_proba(x.numpy())[:, 1], dtype=torch.float32),
    X_scaled, y_raw, axes[0], "Logistic regression"
)
plot_decision_boundary(model_qc,  X_scaled, y_raw, axes[1], f"Q-C  (acc={acc_qc:.2f})")
plot_decision_boundary(model_cqc, X_scaled, y_raw, axes[2], f"C-Q-C (acc={acc_cqc:.2f})")

plt.suptitle("Decision boundaries: moons dataset", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

---
## Demo 3. Encoding strategy comparison: angle vs re-uploading

We compare two encoding strategies on the same 2D dataset:

1. **Single-pass angle encoding**: one $R_y(\pi x_i)$ per qubit, one trainable layer
2. **Re-uploading**: data injected twice, interleaved with trainable layers

Re-uploading allows the circuit to learn functions with higher Fourier frequencies
of the input, which is necessary for more complex decision boundaries.

In [ ]:
# Use circles dataset -- requires a more complex boundary than moons
X_circ_raw, y_circ = make_circles(n_samples=200, noise=0.15, factor=0.5, random_state=7)
X_circ = scaler.fit_transform(X_circ_raw)

X_ctr, X_cte, y_ctr, y_cte = train_test_split(X_circ, y_circ, test_size=0.25, random_state=7)
Xct = torch.tensor(X_ctr, dtype=torch.float32)
yct = torch.tensor(y_ctr, dtype=torch.float32)


# ---- Single-pass encoding ----
@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit_single(inputs, weights):
    """Single-pass angle encoding."""
    qml.AngleEmbedding(inputs * np.pi, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


# ---- Re-uploading encoding ----
@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit_reupload(inputs, weights):
    """Re-uploading: encode -> ansatz -> encode -> ansatz."""
    # First upload
    qml.AngleEmbedding(inputs * np.pi, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights[0:1], wires=range(n_qubits))
    # Second upload (same features, different trainable block)
    qml.AngleEmbedding(inputs * np.pi, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights[1:2], wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


n_basic_layers = 2
qlayer_single   = qml.qnn.TorchLayer(circuit_single,   {"weights": (n_basic_layers, n_qubits)})
qlayer_reupload = qml.qnn.TorchLayer(circuit_reupload, {"weights": (n_basic_layers, n_qubits)})

model_single   = nn.Sequential(qlayer_single,   nn.Sigmoid())
model_reupload = nn.Sequential(qlayer_reupload, nn.Sigmoid())

print("Trainable parameters (single):   ",
      sum(p.numel() for p in model_single.parameters()))
print("Trainable parameters (reupload): ",
      sum(p.numel() for p in model_reupload.parameters()))

In [ ]:
print("Training single-pass model on circles:")
h_single   = train_model(model_single,   Xct, yct, n_epochs=80, lr=0.02)

print("\nTraining re-uploading model on circles:")
h_reupload = train_model(model_reupload, Xct, yct, n_epochs=80, lr=0.02)

with torch.no_grad():
    Xcte_t = torch.tensor(X_cte, dtype=torch.float32)
    acc_s = accuracy_score(y_cte, (model_single(Xcte_t).squeeze().numpy() >= 0.5))
    acc_r = accuracy_score(y_cte, (model_reupload(Xcte_t).squeeze().numpy() >= 0.5))

lr_circ = LogisticRegression(max_iter=1000).fit(X_ctr, y_ctr)
acc_lr_c = lr_circ.score(X_cte, y_cte)

print(f"\nCircles dataset results:")
print(f"  Logistic regression : {acc_lr_c:.3f}")
print(f"  Single-pass QNN     : {acc_s:.3f}")
print(f"  Re-uploading QNN    : {acc_r:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

plot_decision_boundary(
    lambda x: torch.tensor(lr_circ.predict_proba(x.numpy())[:, 1], dtype=torch.float32),
    X_circ, y_circ, axes[0], f"Logistic reg. (acc={acc_lr_c:.2f})"
)
plot_decision_boundary(model_single,   X_circ, y_circ, axes[1], f"Single-pass (acc={acc_s:.2f})")
plot_decision_boundary(model_reupload, X_circ, y_circ, axes[2], f"Re-uploading (acc={acc_r:.2f})")

plt.suptitle("Encoding comparison: circles dataset", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(h_single,   label="single-pass")
plt.plot(h_reupload, label="re-uploading")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Training curves: single vs re-uploading")
plt.legend()
plt.tight_layout()
plt.show()

---
## Demo 4. Gradient inspection: checking for vanishing gradients

Before committing to a training run, it is good practice to check gradient norms
at initialization. Near-zero gradients across all parameters indicate a barren
plateau and training will not progress.

We compare gradient norms for two circuit depths to illustrate how depth affects
trainability.

In [ ]:
def grad_norms_at_init(n_qubits, n_layers, n_trials=20):
    """Estimate mean gradient norm at random initialization.

    Args:
        n_qubits: number of qubits
        n_layers: number of strongly entangling layers
        n_trials: number of random initializations to average over

    Returns:
        Mean and std of gradient norms across trials.
    """
    dev_local = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev_local, interface="torch", diff_method="backprop")
    def circ(inputs, weights):
        qml.AngleEmbedding(inputs * np.pi, wires=range(n_qubits), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return qml.expval(qml.PauliZ(0))

    shapes = {"weights": (n_layers, n_qubits, 3)}
    norms = []
    x_fixed = torch.rand(4, n_qubits)
    y_fixed  = torch.randint(0, 2, (4,)).float()

    for _ in range(n_trials):
        layer = qml.qnn.TorchLayer(circ, shapes)
        model_tmp = nn.Sequential(layer, nn.Sigmoid())
        model_tmp.zero_grad()
        loss = nn.BCELoss()(model_tmp(x_fixed).squeeze(), y_fixed)
        loss.backward()
        total_norm = 0.0
        for p in model_tmp.parameters():
            if p.grad is not None:
                total_norm += p.grad.norm().item() ** 2
        norms.append(total_norm ** 0.5)

    return np.mean(norms), np.std(norms)


print("Gradient norm at initialization (n_qubits=2):")
for L in [1, 2, 4, 8]:
    mean, std = grad_norms_at_init(n_qubits=2, n_layers=L, n_trials=20)
    print(f"  L={L:2d} layers  |  mean grad norm = {mean:.5f} ± {std:.5f}")

In [ ]:
# Same experiment with more qubits to illustrate qubit-count scaling
print("Gradient norm at initialization (L=2 layers):")
for n in [2, 4, 6]:
    mean, std = grad_norms_at_init(n_qubits=n, n_layers=2, n_trials=20)
    print(f"  n={n} qubits  |  mean grad norm = {mean:.6f} ± {std:.6f}")

---
## Demo 5. Multi-class hybrid QNN with three observables

For multi-class classification we measure multiple qubits and pass the vector of
expectation values to a classical softmax head.

Architecture: angle encoding $\to$ strongly entangling layers $\to$
$(\langle Z_0\rangle, \langle Z_1\rangle, \langle Z_2\rangle)$ $\to$ `Linear(3, 3)` $\to$ softmax.

Dataset: three-class blobs.

In [ ]:
n_q3 = 3  # three qubits for three-class problem
dev3 = qml.device("default.qubit", wires=n_q3)


@qml.qnode(dev3, interface="torch", diff_method="backprop")
def circuit3(inputs, weights):
    """Three-qubit circuit for multi-class output."""
    qml.AngleEmbedding(inputs * np.pi, wires=range(n_q3), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_q3))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_q3)]


qlayer3 = qml.qnn.TorchLayer(circuit3, {"weights": (2, n_q3, 3)})


class MulticlassQNN(nn.Module):
    """Hybrid QNN for three-class classification."""

    def __init__(self):
        super().__init__()
        self.qnode = qlayer3
        self.head  = nn.Linear(n_q3, 3)  # no softmax: CrossEntropyLoss applies it

    def forward(self, x):
        x = self.qnode(x)   # (batch, 3), values in [-1, 1]
        return self.head(x) # (batch, 3) raw logits


# Three-blob dataset, 3 features reduced to 3 qubits via scaling
X_blob, y_blob = make_blobs(
    n_samples=300, centers=3, n_features=3,
    cluster_std=1.5, random_state=0
)
X_blob_sc = MinMaxScaler().fit_transform(X_blob)

X_btr, X_bte, y_btr, y_bte = train_test_split(
    X_blob_sc, y_blob, test_size=0.25, random_state=0
)

Xbt  = torch.tensor(X_btr, dtype=torch.float32)
ybt  = torch.tensor(y_btr, dtype=torch.long)
Xbte = torch.tensor(X_bte, dtype=torch.float32)

model3 = MulticlassQNN()

optimizer3 = optim.Adam(model3.parameters(), lr=0.02)
loss_fn3   = nn.CrossEntropyLoss()

history3 = []
for epoch in range(80):
    optimizer3.zero_grad()
    logits = model3(Xbt)
    loss3  = loss_fn3(logits, ybt)
    loss3.backward()
    optimizer3.step()
    history3.append(loss3.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1:3d}  loss = {loss3.item():.4f}")

with torch.no_grad():
    pred3 = model3(Xbte).argmax(dim=1).numpy()

acc3 = accuracy_score(y_bte, pred3)
print(f"\nMulti-class QNN test accuracy: {acc3:.3f}")

---
## Demo 6. ZZ-style entangling encoding vs product angle encoding

We compare two encoding circuits for the same 2D input:

1. **Product angle encoding**: $R_y(\pi x_1) \otimes R_y(\pi x_2)$ (no entanglement in encoding)
2. **ZZ-style encoding**: Hadamard + $R_z(2x_i)$ + CX + $R_z(2 x_1 x_2)$

The ZZ encoding injects the pairwise interaction $x_1 x_2$ as a phase, which
makes the kernel sensitive to feature interactions that the product encoding misses.

In [ ]:
@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit_product(inputs, weights):
    """Product angle encoding + HEA ansatz."""
    qml.AngleEmbedding(inputs * np.pi, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit_zz(inputs, weights):
    """ZZ-style entangling encoding + HEA ansatz."""
    # Hadamard layer
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    # Single-feature phases
    for i in range(n_qubits):
        qml.RZ(2.0 * inputs[i], wires=i)
    # Pairwise interaction
    qml.CNOT(wires=[0, 1])
    qml.RZ(2.0 * inputs[0] * inputs[1], wires=1)
    qml.CNOT(wires=[0, 1])
    # Trainable ansatz
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))


qlayer_prod = qml.qnn.TorchLayer(circuit_product, {"weights": (2, n_qubits)})
qlayer_zz   = qml.qnn.TorchLayer(circuit_zz,      {"weights": (2, n_qubits)})

model_prod = nn.Sequential(qlayer_prod, nn.Sigmoid())
model_zz   = nn.Sequential(qlayer_zz,  nn.Sigmoid())

print("Training product-encoding model on circles:")
h_prod = train_model(model_prod, Xct, yct, n_epochs=80, lr=0.02)

print("\nTraining ZZ-encoding model on circles:")
h_zz = train_model(model_zz, Xct, yct, n_epochs=80, lr=0.02)

with torch.no_grad():
    acc_prod = accuracy_score(y_cte, (model_prod(Xcte_t).squeeze().numpy() >= 0.5))
    acc_zz   = accuracy_score(y_cte, (model_zz(Xcte_t).squeeze().numpy()   >= 0.5))

print(f"\nProduct encoding acc: {acc_prod:.3f}")
print(f"ZZ encoding acc:      {acc_zz:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_decision_boundary(model_prod, X_circ, y_circ, axes[0],
                       f"Product enc. (acc={acc_prod:.2f})")
plot_decision_boundary(model_zz,   X_circ, y_circ, axes[1],
                       f"ZZ enc. (acc={acc_zz:.2f})")
plt.suptitle("Product vs ZZ encoding: circles", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

---
## Demo 7. Architecture rules of thumb: summary table

We collected observations from the demos above. The table below provides a
practical guide for architecture and encoding selection.

In [ ]:
summary = [
    ("Low-dim separable",      "Product angle",    "HEA 1-2 layers",   "Q-C"),
    ("Low-dim non-linear",     "Re-uploading",      "BasicEntangler x2","Q-C"),
    ("Low-dim interacting",    "ZZ / entangling",   "HEA 2 layers",     "C-Q-C"),
    ("High-dim tabular",       "Angle + PCA first", "StronglyEntang.",  "C-Q-C"),
    ("Image patches",          "Angle per patch",   "Convolutional QNN","C-Q-C"),
    ("Multi-class (k classes)","Angle",             "StronglyEntang.",  "Q-C, k observables"),
]

print(f"{'Dataset type':<26} {'Encoding':<22} {'Ansatz':<22} {'Pattern'}")
print("-" * 88)
for row in summary:
    print(f"{row[0]:<26} {row[1]:<22} {row[2]:<22} {row[3]}")

---
## Summary

This demo covered the end-to-end hybrid QNN workflow in PennyLane + PyTorch:

- `TorchLayer` wraps any QNode into a standard `nn.Module`. Gradients flow
  through the circuit via `diff_method='backprop'` in simulation.
- Adding a classical linear head (Q-C or C-Q-C pattern) almost always improves
  accuracy without adding substantial classical capacity.
- Encoding choice matters more than ansatz depth for complex boundaries:
  re-uploading and ZZ-style entangling encodings outperform single-pass
  product encodings on non-linearly separable data.
- Always check gradient norms at initialization. If all norms are near zero,
  reduce depth or change initialization before training.
- Always compare against a logistic regression baseline. A hybrid QNN that
  underperforms logistic regression on the same split has a training or
  encoding problem, not an expressibility problem.